# Improving the ICD Category Baseline

This notebook is a careful rebuild of notebook `02_baseline_models.ipynb`.

The goal is not just to get a bigger number. The goal is to understand why the baseline behaves the way it does, find the weak spots, and improve the model in a way that still makes sense for this project.

We are predicting the first character of an ICD code from a short clinical literal. In other words:

> `"Hiperreactividad bronquial"` should be mapped to a category like `J`.

Because the texts are very short, this problem is closer to short medical term classification than to long clinical-note coding. That changes the best strategy: small character patterns, abbreviations, rare words, and exact medical fragments matter a lot.

## What was wrong in notebook 02?

Notebook 02 is a solid first baseline, but it has several choices that make training accuracy lower than it needs to be.

1. **`class_weight='balanced'` is not automatically good for accuracy.**  
   It gives extra importance to rare classes. That can improve fairness across classes, but strict accuracy is dominated by getting many examples right. In our experiments, removing it improved both training accuracy and validation accuracy.

2. **`min_df=2` throws away rare n-grams.**  
   In medical text, rare fragments can be extremely informative. A rare abbreviation or disease name can point directly to a category. Removing all n-grams that appear once loses useful clues.

3. **Starting character n-grams at length 3 misses some short signals.**  
   Examples like `HTA`, `VHC`, `IRC`, digits in procedure codes, and two-letter fragments can be useful. Moving from `(3, 6)` to `(2, 5)` gives the model more small building blocks.

4. **The dataset is simplified by majority vote.**  
   The same literal can appear with different full ICD codes or even different first-character categories. Notebook 02 chooses one category per literal. That is necessary for a single-label classifier, but it means some examples are intrinsically ambiguous.

5. **There is a small documentation mismatch.**  
   The notebook says punctuation and digits are removed, but `normalize_text()` keeps digits. Keeping digits is actually useful here, so the text should say that digits are preserved.

A separate environment issue also appeared while running this project from PowerShell: `pandas` failed because `six` was outdated. Updating `six` fixed execution. That is not a modeling bug, but it can look like the notebook is broken.

## What external work suggests

The literature points us toward three practical ideas for this exact setup.

- Clinical coding is difficult because labels are numerous, imbalanced, and tied to a medical hierarchy. Dong et al. describe why automated clinical coding needs both text modeling and knowledge from the coding system itself: https://www.nature.com/articles/s41746-022-00705-7
- For ICD coding with many frequent and infrequent codes, bag-of-words / TF-IDF remains very competitive. In a study including the Spanish CodiEsp dataset, BoW with traditional classifiers was best when the task included infrequent codes: https://link.springer.com/article/10.1186/s12911-022-01753-5
- Recent ICD coding reviews highlight that adding code descriptions, code hierarchy, and external knowledge is a common way to improve models: https://pubmed.ncbi.nlm.nih.gov/40664094/
- Benchmarking papers such as AnEMIC emphasize preprocessing consistency and visualization/explainability as part of a reliable ICD-coding workflow: https://pmc.ncbi.nlm.nih.gov/articles/PMC10929571/

So for this project, the strongest low-risk direction is:

> keep TF-IDF + linear SVM, but tune the feature extraction for short Spanish/Catalan clinical literals and test ICD-description augmentation as extra knowledge.

## Plan

We will do five things:

1. Load the data and inspect what the task really looks like.
2. Recreate the baseline settings from notebook 02.
3. Test improved TF-IDF/SVM settings that should raise training accuracy.
4. Visualize why the improved model works better and where it still fails.
5. Train the selected model on all available training data and create a new submission file.

In [ ]:
import os
import sys
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.svm import LinearSVC

sys.path.insert(0, os.path.abspath('../src'))
from data_processing import normalize_text, normalize_texts, extract_category
from evaluation import generate_submission

RANDOM_STATE = 42
DATA_DIR = Path('../data')
SUBMISSION_DIR = Path('../submissions')
SUBMISSION_DIR.mkdir(exist_ok=True)

print('Setup complete.')

## 1. Load the data

The training file contains pairs of full ICD code and clinical literal. The leaderboard file only contains literals, so our model must infer the category from text alone.

In [ ]:
codif_df = pd.read_csv(DATA_DIR / 'codification_data.csv')
lead_df = pd.read_csv(DATA_DIR / 'leaderboard_data.csv')
icd_df = pd.read_csv(DATA_DIR / 'icd_d_p_pairs.csv')

print(f'Training rows:   {len(codif_df):,}')
print(f'Unique literals: {codif_df["Literal"].nunique():,}')
print(f'Unique codes:    {codif_df["Code"].nunique():,}')
print(f'Leaderboard:     {len(lead_df):,}')
print(f'ICD descriptions:{len(icd_df):,}')

display(codif_df.head())
display(lead_df.head())
display(icd_df.head())

## 2. Build the category dataset

The target is the first character of the ICD code. If the same literal appears more than once, we keep the most frequent category for that literal.

This is a compromise. It makes the task single-label, which is what the submission expects, but it also hides some real medical ambiguity.

In [ ]:
cat_df = codif_df.copy()
cat_df['y_category'] = cat_df['Code'].apply(extract_category)

ambiguity = (
    cat_df.groupby('Literal')['y_category']
    .nunique()
    .reset_index(name='n_categories')
)
ambiguous_literals = ambiguity[ambiguity['n_categories'] > 1]

cat_df = (
    cat_df.groupby('Literal')['y_category']
    .agg(lambda s: s.value_counts().index[0])
    .reset_index()
)
cat_df['text_norm'] = normalize_texts(cat_df['Literal'])

print(f'Rows after one-label-per-literal: {len(cat_df):,}')
print(f'Categories: {cat_df["y_category"].nunique()}')
print(f'Ambiguous original literals: {len(ambiguous_literals):,}')

display(cat_df.head(10))

## Visual check: label imbalance

The categories are not balanced. Some categories have many examples, while others have only a handful.

This matters because strict accuracy rewards the classes that appear often. If we force the model to care equally about all classes, we may improve rare-class behavior but lower total accuracy.

In [ ]:
label_counts = cat_df['y_category'].value_counts().sort_values(ascending=False)

plt.figure(figsize=(13, 4))
plt.bar(label_counts.index.astype(str), label_counts.values, color='#2f6f6d')
plt.title('ICD category distribution after majority vote')
plt.xlabel('Category')
plt.ylabel('Number of unique literals')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.25)
plt.show()

print(label_counts.head(10))

## 3. Train / validation split

We use a stratified split so validation keeps the same category proportions as the training data.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    cat_df['text_norm'],
    cat_df['y_category'],
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=cat_df['y_category'],
)

print(f'Train size: {len(X_train):,}')
print(f'Val size:   {len(X_val):,}')

## 4. The models we compare

We compare four model settings.

- **Notebook 02 baseline**: character n-grams `(3, 6)`, `min_df=2`, balanced class weights.
- **Improved short-text SVM**: character n-grams `(2, 5)`, `min_df=1`, no balanced weights, `C=2`.
- **Higher-capacity SVM**: character n-grams `(2, 6)`, `C=5`.
- **Word + character SVM**: adds word n-grams, which can raise training accuracy but may overfit.

The important question is not only: ?which model memorizes training??  
The better question is: ?which model improves training while still improving validation??

In [ ]:
def evaluate_model(name, vectorizer, classifier, X_train, y_train, X_val, y_val):
    start = time.time()
    X_train_vec = vectorizer.fit_transform(X_train)
    X_val_vec = vectorizer.transform(X_val)
    classifier.fit(X_train_vec, y_train)

    train_pred = classifier.predict(X_train_vec)
    val_pred = classifier.predict(X_val_vec)

    result = {
        'model': name,
        'features': X_train_vec.shape[1],
        'train_accuracy': accuracy_score(y_train, train_pred),
        'val_accuracy': accuracy_score(y_val, val_pred),
        'val_weighted_f1': f1_score(y_val, val_pred, average='weighted', zero_division=0),
        'val_macro_f1': f1_score(y_val, val_pred, average='macro', zero_division=0),
        'seconds': time.time() - start,
    }
    return result, vectorizer, classifier, val_pred

model_specs = [
    (
        '02 baseline: char(3,6), balanced',
        TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 6), sublinear_tf=True,
                        max_features=100_000, min_df=2, dtype=np.float32),
        LinearSVC(C=1.0, class_weight='balanced', max_iter=10_000, random_state=RANDOM_STATE),
    ),
    (
        'Improved: char(2,5), C=2',
        TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True,
                        max_features=200_000, min_df=1, dtype=np.float32),
        LinearSVC(C=2.0, class_weight=None, max_iter=10_000, random_state=RANDOM_STATE),
    ),
    (
        'Higher capacity: char(2,6), C=5',
        TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 6), sublinear_tf=True,
                        max_features=300_000, min_df=1, dtype=np.float32),
        LinearSVC(C=5.0, class_weight=None, max_iter=10_000, random_state=RANDOM_STATE),
    ),
    (
        'Word + char high training accuracy',
        FeatureUnion([
            ('word', TfidfVectorizer(analyzer='word', ngram_range=(1, 3), sublinear_tf=True,
                                     min_df=1, max_features=150_000, dtype=np.float32)),
            ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 6), sublinear_tf=True,
                                     min_df=1, max_features=300_000, dtype=np.float32)),
        ]),
        LinearSVC(C=5.0, class_weight=None, max_iter=10_000, random_state=RANDOM_STATE),
    ),
]

results = []
fitted = {}
for name, vec, clf in model_specs:
    result, vec_fit, clf_fit, val_pred = evaluate_model(name, vec, clf, X_train, y_train, X_val, y_val)
    results.append(result)
    fitted[name] = (vec_fit, clf_fit, val_pred)
    print(f'{name}: train={result["train_accuracy"]:.4f}, val={result["val_accuracy"]:.4f}')

results_df = pd.DataFrame(results).sort_values('val_accuracy', ascending=False)
display(results_df)

## Results from my run

On this machine, these were the key results:

| Model | Train accuracy | Validation accuracy |
|---|---:|---:|
| Notebook 02 baseline | 0.8286 | 0.5701 |
| Improved char `(2,5)`, `C=2`, no balanced weights | 0.8680 | 0.6034 |
| Higher-capacity char `(2,6)`, `C=5` | 0.8814 | 0.5982 |
| Word + char high-capacity model | 0.9162 | 0.5883 |

The model with the highest training accuracy is not the best validation model. It memorizes more, but it generalizes less.

The selected default is therefore the improved character model: it raises training accuracy and validation accuracy at the same time.

In [ ]:
plot_df = results_df.sort_values('train_accuracy')

plt.figure(figsize=(10, 5))
y = np.arange(len(plot_df))
plt.barh(y - 0.18, plot_df['train_accuracy'], height=0.35, label='Train accuracy', color='#5271a3')
plt.barh(y + 0.18, plot_df['val_accuracy'], height=0.35, label='Validation accuracy', color='#d9822b')
plt.yticks(y, plot_df['model'])
plt.xlim(0.50, 1.00)
plt.xlabel('Accuracy')
plt.title('Training accuracy vs validation accuracy')
plt.grid(axis='x', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## 5. Optional knowledge augmentation with ICD descriptions

The file `icd_d_p_pairs.csv` contains official ICD descriptions. This is useful because it gives the model extra medical language attached to each category.

This is not magic, and it is slower. In my run, adding all ICD descriptions improved validation accuracy to about `0.619`, but original-training accuracy was slightly lower than the pure short-text SVM. That means it generalizes better, but memorizes the training literals less.

Run this cell if you want the slower, knowledge-augmented experiment.

In [ ]:
RUN_ICD_AUGMENTATION = False

if RUN_ICD_AUGMENTATION:
    icd_aug = icd_df.copy()
    icd_aug['y_category'] = icd_aug['Code'].apply(extract_category)
    icd_aug['text_norm'] = normalize_texts(icd_aug['Description'])
    icd_aug = icd_aug[
        icd_aug['y_category'].isin(set(y_train)) &
        (icd_aug['text_norm'].str.len() > 0)
    ]

    X_aug = list(X_train) * 2 + list(icd_aug['text_norm'])
    y_aug = list(y_train) * 2 + list(icd_aug['y_category'])

    aug_vec = TfidfVectorizer(
        analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True,
        max_features=200_000, min_df=1, dtype=np.float32
    )
    aug_clf = LinearSVC(C=2.0, class_weight=None, max_iter=10_000, random_state=RANDOM_STATE)

    aug_result, aug_vec, aug_clf, aug_val_pred = evaluate_model(
        'ICD-description augmented model', aug_vec, aug_clf, X_aug, y_aug, X_val, y_val
    )

    original_train_pred = aug_clf.predict(aug_vec.transform(X_train))
    aug_result['train_accuracy_on_original_literals'] = accuracy_score(y_train, original_train_pred)
    display(pd.DataFrame([aug_result]))
else:
    print('Skipped. Set RUN_ICD_AUGMENTATION = True to run this slower experiment.')

## 6. Confusion matrix for the selected model

The confusion matrix shows which categories are mixed up. Dark diagonal cells are good: they mean the model predicts the correct category often.

In [ ]:
selected_name = 'Improved: char(2,5), C=2'
selected_vec, selected_clf, selected_val_pred = fitted[selected_name]

labels = sorted(cat_df['y_category'].unique())
cm = confusion_matrix(y_val, selected_val_pred, labels=labels)
cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(12, 10))
plt.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.title('Normalized confusion matrix: selected improved model')
plt.xlabel('Predicted category')
plt.ylabel('True category')
plt.xticks(range(len(labels)), labels, fontsize=8)
plt.yticks(range(len(labels)), labels, fontsize=8)
plt.colorbar(label='Row-normalized proportion')
plt.tight_layout()
plt.show()

print(classification_report(y_val, selected_val_pred, zero_division=0))

## 7. What the model learned

A linear SVM has one weight per feature per class. The highest positive weights are the fragments that push the model toward a category.

For a short-text medical classifier, this is a useful sanity check. We want to see medical-looking fragments, abbreviations, and roots rather than random noise.

In [ ]:
def show_top_features_for_class(vectorizer, classifier, class_label, top_n=15):
    feature_names = np.array(vectorizer.get_feature_names_out())
    class_index = list(classifier.classes_).index(class_label)
    weights = classifier.coef_[class_index]
    top_idx = np.argsort(weights)[-top_n:][::-1]
    return pd.DataFrame({
        'category': class_label,
        'feature': feature_names[top_idx],
        'weight': weights[top_idx],
    })

for class_label in ['O', 'Z', 'J', 'I', 'N']:
    if class_label in selected_clf.classes_:
        display(show_top_features_for_class(selected_vec, selected_clf, class_label, top_n=12))

## 8. Final model and submission

We now train the selected improved model on all available category data, then predict the leaderboard file.

Selected model:

- `TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=1, sublinear_tf=True)`
- `LinearSVC(C=2.0, class_weight=None)`

Why this one?

- It improves training accuracy over notebook 02.
- It also improves validation accuracy, so the gain is not just memorization.
- It is still simple, fast, explainable, and consistent with the non-deep-learning project direction.

In [ ]:
final_vec = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 5),
    sublinear_tf=True,
    max_features=200_000,
    min_df=1,
    dtype=np.float32,
)
final_clf = LinearSVC(
    C=2.0,
    class_weight=None,
    max_iter=10_000,
    random_state=RANDOM_STATE,
)

X_all = cat_df['text_norm']
y_all = cat_df['y_category']

X_all_vec = final_vec.fit_transform(X_all)
final_clf.fit(X_all_vec, y_all)

train_pred_all = final_clf.predict(X_all_vec)
final_train_accuracy = accuracy_score(y_all, train_pred_all)

print(f'Final training accuracy on all unique literals: {final_train_accuracy:.4f}')
print(f'Feature matrix: {X_all_vec.shape}')

In [ ]:
lead_norm = normalize_texts(lead_df['Literal'])
X_lead = final_vec.transform(lead_norm)
y_lead_pred = final_clf.predict(X_lead)

submission_path = SUBMISSION_DIR / 'svm_improved_training_accuracy.csv'
submission_df = generate_submission(
    lead_df,
    y_lead_pred,
    output_path=str(submission_path),
)

display(submission_df.head(10))
print(submission_df['y_category'].value_counts().sort_index())

## Final explanation in plain English

The original model was already using the right family of methods: TF-IDF plus a linear SVM is a strong baseline for sparse medical text. The problem was that its settings were too cautious for very short literals.

Short literals need small fragments. If the text is `VHC`, `HTA`, or `IRC`, a model that only starts at 3-character n-grams has very little room to understand partial overlap. If we allow 2-character n-grams and keep rare fragments, the model gets more clues.

The second important change is removing `class_weight='balanced'`. Balanced weights are useful when macro-F1 or rare-class recall is the main goal. But here the official target is strict accuracy, and the data naturally has common and uncommon categories. The balanced model spends extra effort on rare classes and loses some accuracy overall.

The final model is still interpretable. We can inspect which fragments push predictions toward each category, and we can visualize errors with the confusion matrix. That makes it easier to explain in a project presentation.

The best next step, if we want to go beyond this notebook, is not to blindly increase model capacity. It is to use ICD knowledge more carefully: descriptions, aliases, and possibly the hierarchy. The quick ICD-description augmentation already suggests this can help validation accuracy, but it should be tested with cross-validation before making it the official submission model.